# Volume Profile — Bowl & Dome Detection with Recursive POC

BTCUSDT 1m `rising_from_bowl` / `falling_from_dome` detection charts, each overlaid with recursive-POC volume-profile levels (dashed lines + rank table).

In [ ]:
%pip install numpy numba scipy duckdb pandas plotly

In [ ]:
import os
import sys

REPO_URL  = "https://github.com/payamdav/pycrypto2.git"
REPO_NAME = "pycrypto2"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
# ── Config (edit here) — symmetric bowl/dome params entered once ───────────
config = {
    # general
    "asset": "btcusdt",
    "date_from": "2026-04-15 00:00:00",
    "date_to": "2026-04-21 23:59:00",
    "vwap_period": 1,
    # symmetric pattern params (bowl <-> dome pairs share one value)
    "min_pattern_width": 10,        # min_bowl_width  <-> min_dome_width
    "max_pattern_width": 120,       # max_bowl_width  <-> max_dome_width
    "min_pattern_extent_bps": 20.0, # min_bowl_depth_bps <-> min_dome_height_bps
    "extremum_position_limit": 0.8, # bottom_position_limit <-> top_position_limit
    "wall_retrace_limit_bps": 15.0, # peak_drawdown_limit_bps <-> trough_rally_limit_bps
    "max_wall_search_width": 240,   # max_peak_search_width <-> max_trough_search_width
    # volume profile
    "vp_lookback": 1440, "vp_bins": 200, "vp_bps_range": 100.0,
    "vp_kernel_type": "Triangular", "vp_bandwidth": 5,
    "vp_va_pct": 70.0, "vp_min_poc_volume_ratio": 0.1,
}
# ─────────────────────────────────────────────────────────────────────────

GENERAL_KEYS = ["asset", "date_from", "date_to", "vwap_period"]
VP_KEYS = ["vp_lookback", "vp_bins", "vp_bps_range", "vp_kernel_type", "vp_bandwidth",
           "vp_va_pct", "vp_min_poc_volume_ratio"]

# fan out the symmetric params to each detector's native key names
bowl_cfg = {
    **{k: config[k] for k in GENERAL_KEYS},
    "min_bowl_width": config["min_pattern_width"],
    "max_bowl_width": config["max_pattern_width"],
    "min_bowl_depth_bps": config["min_pattern_extent_bps"],
    "bottom_position_limit": config["extremum_position_limit"],
    "peak_drawdown_limit_bps": config["wall_retrace_limit_bps"],
    "max_peak_search_width": config["max_wall_search_width"],
    **{k: config[k] for k in VP_KEYS},
}

dome_cfg = {
    **{k: config[k] for k in GENERAL_KEYS},
    "min_dome_width": config["min_pattern_width"],
    "max_dome_width": config["max_pattern_width"],
    "min_dome_height_bps": config["min_pattern_extent_bps"],
    "top_position_limit": config["extremum_position_limit"],
    "trough_rally_limit_bps": config["wall_retrace_limit_bps"],
    "max_trough_search_width": config["max_wall_search_width"],
    **{k: config[k] for k in VP_KEYS},
}

print("bowl_cfg:", bowl_cfg)
print("dome_cfg:", dome_cfg)

In [ ]:
import importlib.util


def load_script_module(mod_name, rel_path):
    """Load a scripts/tests/* module by path (script folders are not packages)."""
    path = os.path.join(REPO_PATH, rel_path)
    spec = importlib.util.spec_from_file_location(mod_name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


bowl_mod = load_script_module("rising_from_bowl_test", "scripts/tests/rising_from_bowl/rising_from_bowl_test.py")

res_bowl = bowl_mod.analyze(bowl_cfg)
fig_bowl = bowl_mod.build_figure(res_bowl, bowl_cfg)
fig_bowl.show()

In [ ]:
dome_mod = load_script_module("falling_from_dome_test", "scripts/tests/falling_from_dome/falling_from_dome_test.py")

res_dome = dome_mod.analyze(dome_cfg)
fig_dome = dome_mod.build_figure(res_dome, dome_cfg)
fig_dome.show()

## What to check

- Bowl chart: dotted parabolic fits mark detected bowls; dome chart: dotted parabolic fits mark detected domes.
- Dashed POC lines sit near the range end, within ±`vp_bps_range` bps of the last price.
- Table ranks (row 1, 2, …) match the color of their POC line; POC 1 is the strongest (highest-volume) level.